# Fitted Distributions of Vulnerability for one cell

## Import libraries and functions

In [ ]:
import pandas as pd
import glob
import os
import numpy as np
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
from math import ceil
from sklearn.metrics import r2_score

def plot_correl(x,y,xtitle,ytitle,salida_pdf,ticks_fix=[]):
  # r Pearson
  #r=np.corrcoef(x,y)
  # r2 Coeficiente de determinación
  r2=r2_score(x,y)
  redondealen=1
  maximo=np.max([np.max(x),np.max(y)])
  if len(ticks_fix)==0:
      lensq=ceil(maximo/redondealen)*redondealen
  else:
      lensq=np.max(ticks_fix)
      minval=np.min(ticks_fix)
  plt.figure(figsize=(5, 5))
  #plt.axis('square')
  plt.xlim([minval,lensq])
  plt.ylim([minval,lensq])
  plt.gca().set_xbound(minval, lensq)
  plt.gca().set_ybound(minval, lensq)
  plt.gca().set_aspect('equal')
  if len(ticks_fix)>0:
    plt.xticks(ticks_fix)
    plt.yticks(ticks_fix)
  #plt.gca().set_box_aspect(1)
  box_style=dict(boxstyle='round', facecolor='white', alpha=1)
  plt.rc('font', size=14)
  sepx=0.65
  sepy=0.1
  plt.text(sepx * (lensq - minval) + minval, sepy * (lensq-minval) + minval, "r$^2$="+str(round(r2,4)),{'weight':'heavy','size':14},bbox=box_style)
  plt.grid(color='gray', linestyle='--', linewidth=0.5)
  plt.ylabel(ytitle)
  plt.xlabel(xtitle)
  # Línea de tendencia
  z = np.polyfit(x, y, 1)
  p = np.poly1d(z)
  plt.plot(x,p(x),"r--")
  # Gráficar los puntos
  plt.scatter(x,y,color="blue")
  plt.savefig(salida_pdf,bbox_inches='tight')
  plt.clf()
  
def calcula_ticks(valor_mínimo,valor_máximo,espacios):
  longitud=valor_máximo-valor_mínimo
  tick=np.round(longitud/espacios,2)
  return np.linspace(valor_mínimo,valor_mínimo+tick*espacios,espacios+1)

## Open vulnerabilities

In [2]:
exportar=["IVG"]
variables=pd.read_csv("results_241_653.csv")
variables_filt=variables[variables["Variable"].isin(exportar)]
años=list(variables_filt)[3:37]
variables_filt=variables_filt[años]
vul=variables_filt[años]
vul.head()

,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,...,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035
0,0.79126,0.80749,0.80847,0.78543,0.80791,0.82591,0.79182,0.83087,0.80940,0.80657,...,0.81809,0.79527,0.80593,0.77941,0.79058,0.74750,0.79616,0.79147,0.80097,0.81060
1,0.80337,0.82552,0.81908,0.80682,0.78861,0.79462,0.78685,0.80804,0.82496,0.82355,...,0.80686,0.79310,0.78826,0.78248,0.78646,0.75575,0.78476,0.79263,0.80808,0.76098
2,0.78558,0.81624,0.80039,0.83194,0.80658,0.81997,0.80570,0.76951,0.79578,0.79776,...,0.80248,0.82310,0.78584,0.80413,0.79929,0.78933,0.78469,0.77795,0.77839,0.73303
3,0.82424,0.79039,0.79934,0.81759,0.81400,0.80011,0.85123,0.82772,0.83842,0.77665,...,0.79900,0.80144,0.78407,0.79274,0.79840,0.79592,0.82141,0.79945,0.80026,0.77101
4,0.78735,0.82981,0.82327,0.79464,0.82511,0.78495,0.80842,0.77515,0.81566,0.81465,...,0.77971,0.80732,0.80826,0.80913,0.80145,0.76885,0.78115,0.79731,0.78524,0.79645


## Plot correlations

In [3]:
path_output="../../vulnerability/cor/"

In [4]:
exportar=["IVG"]
variables_filt=variables[variables["Variable"].isin(exportar)]
años=list(variables_filt)[3:37]
distribuciones=[[stats.beta,"beta"],
                [stats.norm,"norm"],
                [stats.expon,"expon"],
                [stats.gumbel_r,"gumbel_r"],
                [stats.gamma,"gamma"]
                ]
for n in range(len(años)):
  var_año=variables_filt[años[n]].dropna()
  for distrib in distribuciones:
    params=distrib[0].fit(var_año)
    fix_ticks=calcula_ticks(0.7,0.9,4)
    if len(params)>0:
      var_sorted=sorted(var_año)
      incremento=0.0001
      percentiles = np.linspace(0+incremento, 1, len(var_sorted)+1)
      fitted_var=distrib[0].ppf(percentiles[:-1],*params)
      for filetype in ["pdf","svg","eps"]:
        plot_correl(fitted_var,
                  var_sorted,
                "Fitted vulnerability",
                "Calculated vulnerability",
                path_output+exportar[0]+"_"+distrib[1]+"_"+str(años[n])+"cor."+filetype,
                fix_ticks
                )

/tmp/ipykernel_110109/1858625644.py:26: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure(figsize=(5, 5))


<Figure size 500x500 with 0 Axes>

<Figure size 500x500 with 0 Axes>

<Figure size 500x500 with 0 Axes>

<Figure size 500x500 with 0 Axes>

...

<Figure size 500x500 with 0 Axes>

<Figure size 500x500 with 0 Axes>

<Figure size 500x500 with 0 Axes>

<Figure size 500x500 with 0 Axes>

<Figure size 500x500 with 0 Axes>